In [1]:
import shutil, os
import pandas as pd
import time
import re
from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import Select

from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

# %%
# Clear the cached drivers
wdm_cache = os.path.expanduser("~/.wdm")
shutil.rmtree(wdm_cache, ignore_errors=True)
print("Cache cleared")

ModuleNotFoundError: No module named 'selenium'

In [ ]:
def accept_cookies(driver, timeout=8):
    try:
        WebDriverWait(driver, timeout).until(
            EC.element_to_be_clickable(
                (By.ID, "CybotCookiebotDialogBodyLevelButtonLevelOptinAllowAll")
            )
        ).click()
        print("Cookies accepted")
    except:
        pass


def remove_editorial_intro(paragraphs):
    if not paragraphs:
        return paragraphs

    while paragraphs:
        first = paragraphs[0].lower()

        editorial_patterns = [
            r"\bin (his|her) christmas broadcast",
            r"\bthe (king|queen)\b.{0,60}\b(broadcast|message|address|spoke|said)\b",
            r"\bthe following is the text",
            r"\bthe broadcast was",
            r"\bthis was the first",
            r"\bdelivered on\b",
            r"\baddressed the nation\b",
            r"\bchristmas broadcast\b.{0,60}\b(watched|aired|televised|recorded)\b",
            r"^(his|her) majesty",
            r"\brecorded (his|her) christmas\b",
            r"\bwere at the start of\b",
            r"\broyal yacht\b",
            r"\bgovernment house\b",
        ]

        if any(re.search(p, first) for p in editorial_patterns):
            paragraphs = paragraphs[1:]
        else:
            break

    return paragraphs


def extract_speech_body(driver):
    soup = BeautifulSoup(driver.page_source, "html.parser")

    containers = ["div.chapters", "div.field--name-body", "article", "main"]

    for selector in containers:
        container = soup.select_one(selector)
        if not container:
            continue

        for junk in container.select("figure, script, style, nav, header, footer"):
            junk.decompose()

        # Remove italic intro descriptions
        for tag in container.select("p em, p i"):
            parent = tag.find_parent("p")
            if parent and parent == container.find("p"):
                parent.decompose()
                break

        paragraphs = [
            p.get_text(strip=True)
            for p in container.find_all("p")
            if p.get_text(strip=True)
        ]

        paragraphs = remove_editorial_intro(paragraphs)

        if paragraphs:
            return "\n\n".join(paragraphs)

    return None


def extract_title_and_date(driver):
    """Extract title and date from the speech page itself."""
    soup = BeautifulSoup(driver.page_source, "html.parser")

    # --- Title ---
    title = None

    # Try common heading selectors
    for selector in ["h1.page-title", "h1.article-header__title", "h1", "title"]:
        el = soup.select_one(selector)
        if el:
            title = el.get_text(strip=True)
            # Remove site name suffix if present (e.g. " | The Royal Family")
            title = re.sub(r"\s*[|\-–]\s*(The )?Royal.*$", "", title, flags=re.IGNORECASE).strip()
            if title:
                break

    # --- Date ---
    date_str = None
    date_obj = None

    # Try <time> element first
    time_el = soup.find("time")
    if time_el:
        # Prefer datetime attribute
        dt_attr = time_el.get("datetime", "")
        if dt_attr:
            date_str = dt_attr
        else:
            date_str = time_el.get_text(strip=True)

    # Try meta tags
    if not date_str:
        for meta_name in ["article:published_time", "date", "DC.date"]:
            meta = soup.find("meta", {"property": meta_name}) or soup.find("meta", {"name": meta_name})
            if meta and meta.get("content"):
                date_str = meta["content"]
                break

    # Try common date class selectors
    if not date_str:
        for selector in [".date", ".article-date", ".published-date", "span.date", "p.date"]:
            el = soup.select_one(selector)
            if el:
                date_str = el.get_text(strip=True)
                break

    # Parse the date string into a datetime object
    if date_str:
        date_str_clean = date_str.strip().split("T")[0]  # handle ISO format
        for fmt in ["%Y-%m-%d", "%d %B %Y", "%B %d, %Y", "%d/%m/%Y", "%d %b %Y"]:
            try:
                date_obj = datetime.strptime(date_str_clean, fmt)
                break
            except ValueError:
                continue

        # If still not parsed, try extracting with regex
        if not date_obj:
            m = re.search(r"(\d{1,2})\s+(\w+)\s+(\d{4})", date_str)
            if m:
                try:
                    date_obj = datetime.strptime(f"{m.group(1)} {m.group(2)} {m.group(3)}", "%d %B %Y")
                except ValueError:
                    pass

    return title, date_obj


def set_date_filter(driver, start_date_str, end_date_str):
    """Set the from/to date filters using stable data-drupal-selector attributes."""
    try:
        from_input = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "input[data-drupal-selector='edit-dates-min']")
            )
        )
        from_input.clear()
        from_input.send_keys(start_date_str)
        from_input.send_keys(Keys.TAB)
        time.sleep(0.3)
    except Exception as e:
        print(f"  Warning: could not set min date: {e}")

    if end_date_str:
        try:
            to_input = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located(
                    (By.CSS_SELECTOR, "input[data-drupal-selector='edit-dates-max']")
                )
            )
            to_input.clear()
            to_input.send_keys(end_date_str)
            to_input.send_keys(Keys.TAB)
            time.sleep(0.3)
        except Exception as e:
            print(f"  Warning: could not set max date: {e}")


def select_monarch_filter(driver, monarch_value):
    """
    Select a monarch from the #edit-mrf <select> dropdown by option value.
    monarch_value: the option value string from the site HTML:
      '2'  = Queen Elizabeth II
      '3'  = The King (formerly The Prince of Wales) = King Charles III
    """
    try:
        select_el = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "edit-mrf"))
        )
        sel = Select(select_el)
        sel.select_by_value(monarch_value)
        selected_text = sel.first_selected_option.text
        print(f"  Selected monarch: {selected_text}")
    except Exception as e:
        print(f"  Warning: could not select monarch filter: {e}")


def apply_filters_and_search(driver):
    """
    Click the submit button that lives in the same form as the date inputs.
    We find the parent form via the stable data-drupal-selector on the date
    field, then click its submit button — this avoids accidentally triggering
    the site search form which shares the same page.
    """
    try:
        # Find the filter form via the stable date input inside it
        date_input = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "input[data-drupal-selector='edit-dates-min']")
            )
        )
        # Walk up to the parent <form>
        filter_form = date_input.find_element(By.XPATH, "./ancestor::form[1]")
        # Find the submit button scoped to that form only
        submit_btn = filter_form.find_element(
            By.CSS_SELECTOR, "input[type='submit']"
        )
        driver.execute_script("arguments[0].scrollIntoView(true);", submit_btn)
        time.sleep(0.3)
        driver.execute_script("arguments[0].click();", submit_btn)
        print("  Filter form submit button clicked")
        time.sleep(4)
    except Exception as e:
        print(f"  Warning: could not submit filter form: {e}")


def collect_speech_urls(driver):
    """
    Paginate through the listing and collect all speech URLs.
    Filters are applied once before calling this function.
    We simply follow the pager next-link directly — Drupal encodes
    the active filter state into the pager hrefs automatically.
    """
    speech_urls = {}
    page_num = 1

    while True:
        print(f"  Scanning listing page {page_num}")
        time.sleep(2)

        soup = BeautifulSoup(driver.page_source, "html.parser")

        found_on_page = 0
        for a in soup.find_all("a", href=True):
            href = a["href"]
            title = a.get_text(strip=True)

            if href.startswith("/") and re.search(r"/(speeches|broadcast|address|message)/", href, re.IGNORECASE):
                full_url = "https://www.royal.uk" + href
                if full_url not in speech_urls:
                    speech_urls[full_url] = title
                    found_on_page += 1

            elif href.startswith("/") and re.search(r"\b(19|20)\d{2}\b", href):
                if not any(skip in href for skip in ["/news/", "/photos/", "/video/", "/tags/"]):
                    full_url = "https://www.royal.uk" + href
                    if full_url not in speech_urls:
                        speech_urls[full_url] = title
                        found_on_page += 1

        print(f"    Found {found_on_page} new URLs on this page (total: {len(speech_urls)})")

        try:
            next_href = driver.find_element(
                By.CSS_SELECTOR, "li.pager__item--next a"
            ).get_attribute("href")
            print(f"  Next page URL: {next_href}")
            driver.get(next_href)
            page_num += 1
        except:
            print("  No more pages.")
            break

    return speech_urls

In [ ]:
options = Options()
options.add_argument("--incognito")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()), options=options
)
driver.set_window_size(1400, 900)


START_URL = "https://www.royal.uk/media-centre/speeches"





MONARCHS_CONFIG = [
    {
        "monarch": "King Charles III",
        "filter_value": "3",                
        "start_date": "08/09/2022",        
        "end_date": "",                      
    },
    {
        "monarch": "Queen Elizabeth II",
        "filter_value": "2",                
        "start_date": "06/02/1952",          
        "end_date": "08/09/2022",            
    },
]

all_records = []

for config in MONARCHS_CONFIG:
    monarch_name = config["monarch"]
    print(f"\n{'='*60}")
    print(f"Scraping speeches for: {monarch_name}")
    print(f"{'='*60}")

    # Navigate (fresh) to the speeches page
    driver.get(START_URL)
    accept_cookies(driver)
    time.sleep(3)

    # Set date range
    print(f"  Setting date range: {config['start_date']} → {config['end_date'] or 'present'}")
    set_date_filter(driver, config["start_date"], config["end_date"])

    # Select monarch filter
    print(f"  Selecting monarch filter for: {config['monarch']}")
    select_monarch_filter(driver, config["filter_value"])

    # Apply / submit filters
    apply_filters_and_search(driver)

    # Collect all speech URLs across pages
    speech_urls = collect_speech_urls(driver)
    print(f"  Collected {len(speech_urls)} candidate pages for {monarch_name}")




    visited_pages = set()

    for i, (url, link_text) in enumerate(speech_urls.items(), 1):
        print(f"  [{i}/{len(speech_urls)}] {link_text[:80]}")

        try:
            driver.get(url)

            # Canonical URL dedupe
            canonical = driver.current_url.split("?")[0].rstrip("/")
            if canonical in visited_pages:
                print("    Duplicate alias skipped")
                continue
            visited_pages.add(canonical)

            # Wait for content to load
            try:
                WebDriverWait(driver, 8).until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "main, article, div.chapters")
                    )
                )
            except:
                pass

            # Extract title and date from the page
            title, date_obj = extract_title_and_date(driver)

            # Fallback: try to get year from URL / link text
            if not date_obj:
                year_match = re.search(r"\b(19|20)\d{2}\b", canonical + link_text)
                if year_match:
                    year = int(year_match.group())
                    date_obj = datetime(year, 1, 1)  # approximate — year only
                    print(f"    Warning: date not found on page, using year {year} from URL")

            if not date_obj:
                print("    No date found — skipped")
                continue

            if not title:
                title = link_text.strip() or f"Speech {date_obj.strftime('%Y-%m-%d')}"

            # Extract speech body
            speech = extract_speech_body(driver)

            if not speech or len(speech.split()) < 80:
                print("    Too short or empty — skipped")
                continue

            all_records.append(
                {
                    "date": date_obj.strftime("%Y-%m-%d"),
                    "year": date_obj.year,
                    "monarch": monarch_name,
                    "title": title,
                    "speech": speech,
                    "url": canonical,
                }
            )

            print(f"    OK — {len(speech.split())} words | {title[:60]}")

        except Exception as e:
            print(f"    Failed: {e}")

driver.quit()


Cache cleared

Scraping speeches for: King Charles III
Cookies accepted
  Setting date range: 08/09/2022 → present
  Selecting monarch filter for: King Charles III
  Selected monarch: The King (formerly The Prince of Wales)
  Filter form submit button clicked
  Scanning listing page 1
    Found 9 new URLs on this page (total: 9)
  Next page URL: https://www.royal.uk/media-centre/speeches?text=&mrf=3&dates%5Bmin%5D=08/09/2022&dates%5Bmax%5D=26/02/2026&page=1
  Scanning listing page 2
    Found 9 new URLs on this page (total: 18)
  Next page URL: https://www.royal.uk/media-centre/speeches?text=&mrf=3&dates%5Bmin%5D=08/09/2022&dates%5Bmax%5D=26/02/2026&page=2
  Scanning listing page 3
    Found 8 new URLs on this page (total: 26)
  Next page URL: https://www.royal.uk/media-centre/speeches?text=&mrf=3&dates%5Bmin%5D=08/09/2022&dates%5Bmax%5D=26/02/2026&page=3
  Scanning listing page 4
    Found 10 new URLs on this page (total: 36)
  Next page URL: https://www.royal.uk/media-centre/speeches

In [ ]:
df = pd.DataFrame(all_records).sort_values(["monarch", "date"])
df.to_csv("all_UK_monarch_speeches.csv", index=False)

print("\nDONE")
print(f"Saved {len(df)} speeches to all_UK_monarch_speeches.csv")
print(df.groupby("monarch")["title"].count().to_string())
